In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# <span style="color:red"> 코로나 기간 정상화 학습모델링 전처리데이터 </span>

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from keras.models import Sequential
from keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# 1. Load & preprocess
df = pd.read_csv(r'C:\ai_x2\source\proz2\외국인입국자_전처리완료_딥러닝용.csv', parse_dates=[0])
df.columns = ['ds', 'y']
df = df.sort_values('ds')
df = df.dropna()
df['y'] = df['y'].astype(float)

# 기준년도: 팬데믹 이전
train = df[df['ds'] < '2020']
predict_period = df[(df['ds'] >= '2020') & (df['ds'] <= '2023')]

# 결과 저장용
results = pd.DataFrame({'ds': predict_period['ds']})
model_outputs = {}

# 2. Prophet
model = Prophet()
model.fit(train)
future = model.make_future_dataframe(periods=len(predict_period), freq='D')
forecast = model.predict(future)
results['prophet'] = forecast[-len(predict_period):]['yhat'].values
model_outputs['Prophet'] = results['prophet']

# 3. SARIMA
sarima = SARIMAX(train['y'], order=(1,1,1), seasonal_order=(1,1,1,12))
sarima_fit = sarima.fit()
sarima_forecast = sarima_fit.forecast(len(predict_period))
results['sarima'] = sarima_forecast.values
model_outputs['SARIMA'] = results['sarima']

# 4. Random Forest
train_rf = train.copy()
train_rf['ds_ordinal'] = train_rf['ds'].map(pd.Timestamp.toordinal)
rf_model = RandomForestRegressor(n_estimators=100)
rf_model.fit(train_rf[['ds_ordinal']], train_rf['y'])

predict_rf = predict_period.copy()
predict_rf['ds_ordinal'] = predict_rf['ds'].map(pd.Timestamp.toordinal)
results['rf'] = rf_model.predict(predict_rf[['ds_ordinal']])
model_outputs['RandomForest'] = results['rf']

# 5. XGBoost
xgb_model = XGBRegressor()
xgb_model.fit(train_rf[['ds_ordinal']], train_rf['y'])
results['xgb'] = xgb_model.predict(predict_rf[['ds_ordinal']])
model_outputs['XGBoost'] = results['xgb']

# 6. LSTM
scaler = MinMaxScaler()
scaled_y = scaler.fit_transform(train['y'].values.reshape(-1,1))

X, y_lstm = [], []
for i in range(30, len(scaled_y)):
    X.append(scaled_y[i-30:i])
    y_lstm.append(scaled_y[i])
X, y_lstm = np.array(X), np.array(y_lstm)

lstm_model = Sequential()
lstm_model.add(LSTM(64, activation='relu', input_shape=(X.shape[1], 1)))
lstm_model.add(Dense(1))
lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.fit(X, y_lstm, epochs=20, verbose=0)

# 예측
last_seq = scaled_y[-30:]
lstm_preds = []
for _ in range(len(predict_period)):
    pred = lstm_model.predict(last_seq.reshape(1,30,1), verbose=0)
    lstm_preds.append(pred[0][0])
    last_seq = np.append(last_seq[1:], pred, axis=0)

results['lstm'] = scaler.inverse_transform(np.array(lstm_preds).reshape(-1,1)).flatten()
model_outputs['LSTM'] = results['lstm']

# 7. 평균 모델 기반 데이터 재생성
results['recovered'] = results[['prophet','sarima','rf','xgb','lstm']].mean(axis=1)

# 8. 팬데믹 기간에 대체 적용
recovered_df = df.copy()
recovered_df.loc[(recovered_df['ds'] >= '2020') & (recovered_df['ds'] <= '2023'), 'y'] = results['recovered'].values

# 9. 저장
recovered_df.to_csv('/mnt/data/recovered_normal_data.csv', index=False)

# 10. 시각화
plt.figure(figsize=(14,5))
plt.plot(df['ds'], df['y'], label='Original')
plt.plot(recovered_df['ds'], recovered_df['y'], label='Recovered (Normal)', linestyle='--')
plt.axvspan(pd.to_datetime('2020'), pd.to_datetime('2023'), color='gray', alpha=0.3, label='Pandemic')
plt.legend()
plt.title("Actual vs Recovered Normal Demand")
plt.tight_layout()
plt.show()

ValueError: Length mismatch: Expected axis has 15 elements, new values have 2 elements